# 모듈·패키지: 기능의 출처와 인터페이스를 확인하기

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 이전 주차의 변수·자료형과 셀 실행 방법

## 오늘 설명할 수 있어야 하는 것

- 모듈 파일과 import의 관계를 설명한다.
- 필요한 파일·함수 이름·검색 경로를 AI에 전달하고 실제 기능을 확인한다.


## 1. 먼저 원리를 읽기

모듈은 재사용할 코드를 담은 `.py` 파일입니다. 패키지는 관련 모듈을 묶습니다. `import`가 성공하려면 파일과 검색 경로가 맞아야 합니다.
이 실습은 필요한 모듈을 임시 폴더에 직접 생성합니다. 원본 노트북이 요구하던 개인 Drive와 누락된 `mod1.py`를 준비할 필요가 없습니다.


## 2. 실행 전 예측하기

`mod1.add(3, 4)`와 `mod1.sub(3, 4)`는 각각 무엇을 반환할까요? 모듈이 반환값 대신 print만 하면 어떻게 달라질까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
from pathlib import Path
import tempfile
import importlib.util

module_dir = Path(tempfile.mkdtemp(prefix="ce100-module-"))
module_path = module_dir / "mod1.py"
module_path.write_text(
    "def add(a, b):\n    return a + b\n\ndef sub(a, b):\n    return a - b\n",
    encoding="utf-8",
)
spec = importlib.util.spec_from_file_location("ce100_mod1", module_path)
mod1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod1)
print("불러온 파일:", mod1.__file__)
assert mod1.add(3, 4) == 7 and mod1.sub(3, 4) == -1


### 관찰을 원리와 연결하기

일반적인 `import mod1`은 `sys.path`에서 파일을 찾습니다. 여기서는 임시 폴더의 정확한 파일을 지정해 다른 실습의 파일과 섞이지 않도록 했습니다. 모듈을 불러오는 방식은 준비 코드로 읽고, 함수 계약 검토에 집중하세요.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
package_dir = module_dir / "my_package"
package_dir.mkdir(exist_ok=True)
(package_dir / "__init__.py").write_text("", encoding="utf-8")
(package_dir / "my_module.py").write_text(
    "def say_hello(name):\n    return f'Hello, {name}'\n", encoding="utf-8"
)
print("파일 구조:", sorted(str(p.relative_to(module_dir)) for p in module_dir.rglob("*.py")))
import sys
# 검색 경로를 잠시 추가하고 실습 후 원래대로 돌립니다.
sys.path.insert(0, str(module_dir))
try:
    from my_package import my_module
    assert my_module.say_hello("A") == "Hello, A"
    print(my_module.say_hello("A"))
finally:
    sys.path.pop(0)


## 3. 작은 기능을 직접 완성하기

`get_add(module)`은 모듈 객체를 받아, 이름이 `add`인 **호출 가능한 객체**를 반환합니다. 없거나 호출할 수 없으면 `ValueError`를 냅니다.
함수가 있다는 사실만으로 동작까지 맞다고 할 수 있을까요? 반환받은 함수에 `(2, 3)`을 넣는 시험도 수행하세요.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def get_add(module):
    # TODO: getattr, callable의 도움말을 확인하고 인터페이스를 검사하세요.
    raise NotImplementedError("모듈의 인터페이스를 확인하세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    from types import SimpleNamespace
    add = candidate(SimpleNamespace(add=lambda a, b: a + b))
    assert callable(add) and add(2, 3) == 5
    for invalid in [SimpleNamespace(), SimpleNamespace(add=5)]:
        try:
            candidate(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("없거나 호출 불가능한 add를 거절하세요")
    print("공개 점검 통과: 이름 존재·호출 가능 여부")

try:
    check_answer(get_add)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
from types import SimpleNamespace
fake_module = SimpleNamespace(add=5)
print("이름만 확인:", hasattr(fake_module, "add"))
print("호출 가능한가:", callable(fake_module.add))
assert hasattr(fake_module, "add") and not callable(fake_module.add)


### 검토 질문

이름의 존재와 기능의 존재는 다릅니다. 또 `callable`이어도 더하기를 수행한다는 보장은 없습니다. 인터페이스 검사와 행동 검사를 구분해 지시하세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

실제 모듈 파일명과 사용할 함수의 계약을 전달한다. 설치 명령을 추측하지 말고 표준 라이브러리인지 외부 패키지인지 먼저 확인하게 하며, 모듈 출처·호출 가능 여부·반환값을 검사하도록 요청한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

AI가 존재하지 않는 라이브러리 함수를 제안하면 어떤 순서로 확인할까요? import 성공만으로 충분하지 않은 이유를 적으세요.


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 선택 확장

동일 이름의 다른 파일이 검색 경로 앞에 있으면 어떻게 될지 설명하세요. 패키지 내부 모듈의 인터페이스도 같은 방식으로 검토합니다.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
